# Local reconstruction confirmation: Kaggle runner

This is an alternative execution entry point for the same confirmation study, not a new experiment. Keep this notebook and its input dataset **private**. Select **GPU T4 x2** (the code uses one GPU), enable Internet, and attach `calolab-reco-confirmation.bundle` as a private input dataset. The `.bundle` file is the unchanged input ZIP with a different extension so Kaggle does not automatically extract it.

Use **Save Version > Save & Run All** for a background run. Estimated time on one T4: 2-2.5 hours, not yet measured for this complete study. Training plus selection has the same cumulative three-hour ceiling. No reserved-test events are included. GPU availability and quota depend on the account and current capacity.


In [ ]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

SMOKE = os.environ.get("CALOLAB_CONFIRMATION_SMOKE") == "1"
BUNDLE_SHA256 = "0e758bf20eda5cde456ff77ca218a396705437bff24cff63c718484e24509fe9"
ROOT = Path(os.environ.get("CALOLAB_CONFIRMATION_ROOT", "/kaggle/temp/calolab-confirmation"))
OUTPUT = Path(os.environ.get("CALOLAB_CONFIRMATION_OUTPUT", "/kaggle/working"))
ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
DEVICE = "cpu" if SMOKE else "cuda"
# For a restart, attach the saved output as input and set this to its run directory.
PREVIOUS_RUN = None
print("Mode:", "tiny CPU smoke, not scientific" if SMOKE else "full GPU confirmation")


def run_command(command, **kwargs):
    """Stream both output channels into notebook logs, including the actual error."""
    with subprocess.Popen(
        list(map(str, command)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, **kwargs,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait()
    if code:
        raise RuntimeError(f"Command failed with exit code {code}; see the output above.")

## Input and persistence

The input checksum must match the original Colab bundle. Code, environment and input extraction go under `/kaggle/temp`; checkpoints and exports go under `/kaggle/working`. There is no Google Drive mount.

**Working files are not an independent remote backup.** Save a version with outputs and download the result ZIP. For an interactive run, select file persistence and save outputs before ending the session; an unexpected loss can still lose work. To resume, attach a previously saved output as input and set `PREVIOUS_RUN` above to its `calolab-confirmation/<bundle-id>` directory. Checkpoints are validated by the unchanged training code before reuse. Do not combine runs from other bundles.


In [ ]:
configured = os.environ.get("CALOLAB_CONFIRMATION_BUNDLE")
if configured:
    candidates = [Path(configured)]
else:
    candidates = sorted(Path("/kaggle/input").rglob("calolab-reco-confirmation.bundle"))
    candidates += sorted(Path("/kaggle/input").rglob("calolab-reco-confirmation.zip"))
matching = [p for p in candidates if hashlib.sha256(p.read_bytes()).hexdigest() == BUNDLE_SHA256]
if len(matching) != 1:
    raise RuntimeError("Attach exactly one matching confirmation .bundle file as private input.")
bundle = matching[0]
print("Verified input:", bundle.name, BUNDLE_SHA256)

In [ ]:
# Bootstrap only after verifying the complete input ZIP against the fixed checksum.
workspace = ROOT / "workspaces" / BUNDLE_SHA256
workspace.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(bundle) as archive:
    manifest = json.loads(archive.read("MANIFEST.json"))
    if set(archive.namelist()) != set(manifest["files"]) | {"MANIFEST.json"}:
        raise RuntimeError("Unexpected input members.")
    for name, expected in manifest["files"].items():
        target = (workspace / name).resolve()
        if not target.is_relative_to(workspace.resolve()):
            raise RuntimeError("Unsafe archive path.")
        value = archive.read(name)
        if hashlib.sha256(value).hexdigest() != expected:
            raise RuntimeError("Input member checksum differs.")
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists() and target.read_bytes() != value:
            raise RuntimeError("Workspace differs; use a fresh workspace.")
        target.write_bytes(value)
(workspace / "INPUT_MANIFEST.json").write_text(json.dumps(manifest))
if SMOKE:
    python = Path(os.environ["CALOLAB_CONFIRMATION_PYTHON"])
else:
    if shutil.which("uv") is None:
        run_command([sys.executable, "-m", "pip", "install", "uv==0.12.6"])
    run_command(
        ["uv", "sync", "--locked", "--extra", "cuda", "--no-dev", "--python", "3.12"],
        cwd=workspace,
    )
    python = workspace / ".venv/bin/python"
env = dict(
    os.environ,
    PYTHONPATH=str(workspace / "src"),
    PYTHONUNBUFFERED="1",
    MPLCONFIGDIR=str(ROOT / "matplotlib"),
    MPLBACKEND="Agg",
)


def run_module(module, *args):
    run_command(
        [str(python), "-u", "-m", "calolab_reco.confirmation." + module, *map(str, args)],
        cwd=workspace,
        env=env,
    )


run_dir = OUTPUT / "calolab-confirmation" / (BUNDLE_SHA256[:16] + ("-smoke" if SMOKE else ""))
if PREVIOUS_RUN is not None:
    previous = Path(PREVIOUS_RUN)
    if not previous.is_dir() or not (previous / "protocol.json").is_file():
        raise RuntimeError("Previous run must contain protocol.json and committed checkpoints.")
    if not run_dir.exists():
        shutil.copytree(previous, run_dir)
print("Workspace:", workspace)
print("Checkpoint and report directory:", run_dir)

## Protocol check

The main condition uses three seeds. Each seed has two CNN specialists, two direct Transformer specialists, **one** shared pretraining and two separate fine-tunings. Clean and noise-only controls have one seed and no additional pretraining.

**Reading check:** why does a position offset from the observed maximum not leak the true impact? Because the origin is calculated from measured crystals, without labels. Why keep clean controls? They show whether an advantage depends on the measurement scenario.


In [ ]:
protocol = json.loads((workspace / "configs/confirmation.json").read_text())
print(
    json.dumps(
        {
            k: protocol[k]
            for k in (
                "training_seeds",
                "primary_regime",
                "control_regimes",
                "updates",
                "pretraining_updates",
                "energy_loss",
                "position_loss",
                "reserved_test",
            )
        },
        indent=2,
    )
)
run_command(
    [
        str(python),
        "-c",
        "from calolab_reco.pilot import configure; import json; print(json.dumps(configure('"
        + DEVICE
        + "', 20260925), indent=2))",
    ],
    env=env,
    cwd=workspace,
)

## Verify recovery before full training

Tiny subsets are used only to compare uninterrupted updates with interrupted updates resumed in a new process. We compare weights, optimizer, schedule, sampler, random states and selected checkpoint. A failure stops execution before the full study.


In [ ]:
replay = ROOT / "recovery-checks" / (BUNDLE_SHA256[:16] + "-" + DEVICE)
replay = replay / str(time.time_ns())
run_module("verification", "--workspace", workspace, "--output", replay, "--device", DEVICE)
replay_result = json.loads((replay / "verification.json").read_text())
assert all(value["exact"] for value in replay_result.values())
print("Fresh-process replay passed for pretraining and fine-tuning.")

## Train and compare

Progress is printed every 50 updates and validation every 220 updates. Atomic, checksummed checkpoints stay in the output directory. Rerunning resumes them. A saved Kaggle output or a downloaded ZIP is needed to recover after losing that directory. The protocol, seeds, input identity, task split and training ceiling are unchanged.


In [ ]:
env["MPLBACKEND"] = "Agg"  # Render files without inheriting the notebook display backend.
if not all(value["exact"] for value in replay_result.values()):
    raise RuntimeError("Recovery checks must pass first.")
arguments = ["--workspace", workspace, "--output", run_dir, "--device", DEVICE]
if SMOKE:
    arguments.append("--smoke")
run_module("training", *arguments)
summary = json.loads((run_dir / "summary.json").read_text())
print("Complete:", summary["complete"], "| Recorded phases:", len(summary["cases"]))
print("A ceiling stop is a partial run. Export it for review; do not silently increase the budget.")
(run_dir / "recovery_verification.json").write_text(json.dumps(replay_result, indent=2))

## Read the results

Lower errors are better. Dashed lines are matched classical references. Points are individual training seeds. The fine-tuning curve excludes pretraining; the report also includes that cost and counts shared pretraining once for the two tasks.

**Reading check:** does a faster fine-tuning mean less total computation? Only if its saving also covers pretraining. A strong position result does not imply an energy gain.


In [ ]:
from IPython.display import Image, Markdown, display

run_module("reporting", "--run", run_dir)
report_text = (run_dir / "RESULTS.md").read_text()
for caption, name in [("Task comparisons", "comparison.png"), ("Learning curves", "learning.png")]:
    report_text = report_text.replace(f"![{caption}](figures/{name})", "")
display(Markdown(report_text))
for filename in ("comparison.png", "learning.png"):
    display(Image(filename=str(run_dir / "figures" / filename)))
reconstruction = sorted((run_dir / "figures").glob("*pretraining.png"))
if reconstruction:
    display(Image(filename=str(reconstruction[0])))

## Verify and return the ZIP

The verifier checks input identity, completion, checkpoints and validation predictions against the original IDs and targets. It never evaluates the reserved test. Download the final `calolab-reco-confirmation-results-*.zip` from the saved version's **Output** tab. A ceiling stop is a partial result, not a finished study.


In [ ]:
stamp = time.time_ns()
result_zip = OUTPUT / (
    "calolab-reco-confirmation-" + ("smoke" if SMOKE else "results") + f"-{stamp}.zip"
)
run_module("transport", "export", "--run", run_dir, "--output", result_zip)
run_module("transport", "verify", "--bundle", bundle, "--result", result_zip)
print("Verified export:", result_zip.name)
print("SHA256:", hashlib.sha256(result_zip.read_bytes()).hexdigest())
print("Download this ZIP from the saved version's Output tab.")